

| Secret name | Value |
|---|---|
| `DB_HOST` | e.g. `ep-xxxx.us-east-2.aws.neon.tech` (from Neon/Supabase) |
| `DB_PORT` | usually `5432` |
| `DB_NAME` | your database name |
| `DB_USER` | your database user |
| `DB_PASSWORD` | your database password |
| `JWT_SECRET` | any long random string (generate one in the next cell) |
| `SMTP_EMAIL` | your Gmail address |
| `SMTP_APP_PASSWORD` | 16-character Gmail **App Password** (not your real password) |
| `NGROK_AUTHTOKEN` | from https://dashboard.ngrok.com/get-started/your-authtoken |

**Note:** the FastAPI backend added below reuses `JWT_SECRET` — no additional secrets are needed for it.


In [1]:
!pip install -q streamlit psycopg2-binary PyJWT bcrypt \
    python-dotenv email-validator pyngrok \
    fastapi uvicorn python-multipart requests \
    langdetect ftfy emoji deep-translator vaderSentiment spacy pandas matplotlib \
    transformers accelerate torch stopwordsiso
!python -m spacy download xx_sent_ud_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 24.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('xx_sent_ud_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## New: Employee Wellness NLP Analysis

After login, the app now has an **"Run NLP Analysis"** button next to the file upload. It sends the CSV/TXT to a new `/analyze` endpoint on the FastAPI backend, which detects language (Telugu/Kannada-aware), cleans and tokenizes the text, translates it to English, lemmatizes, and runs VADER sentiment + keyword-based emotion detection. Results (including sentiment/emotion bar charts) render inline in Streamlit.

**No new secrets needed** — this reuses `JWT_SECRET` like the upload feature already did.

**Heads-up:** installing spaCy + downloading the `xx_sent_ud_sm` model adds a few minutes to Section 2's install cell the first time you run it in a fresh Colab runtime.


In [2]:
from google.colab import userdata

required_secrets = [
    "DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD",
    "JWT_SECRET", "SMTP_EMAIL", "SMTP_APP_PASSWORD", "NGROK_AUTHTOKEN",
]

values = {}
missing = []
for key in required_secrets:
    try:
        values[key] = userdata.get(key)
    except Exception:
        missing.append(key)

if missing:
    raise RuntimeError(
        f"Missing Colab secrets: {missing}. "
        f"Add them via the key icon in the left sidebar, then re-run this cell."
    )

env_content = f'''DB_HOST={values["DB_HOST"]}
DB_PORT={values["DB_PORT"]}
DB_NAME={values["DB_NAME"]}
DB_USER={values["DB_USER"]}
DB_PASSWORD={values["DB_PASSWORD"]}

JWT_SECRET={values["JWT_SECRET"]}
JWT_ALGORITHM=HS256
JWT_EXPIRY_MINUTES=60

SMTP_HOST=smtp.gmail.com
SMTP_PORT=587
SMTP_EMAIL={values["SMTP_EMAIL"]}
SMTP_APP_PASSWORD={values["SMTP_APP_PASSWORD"]}

OTP_EXPIRY_MINUTES=10
'''

with open(".env", "w") as f:
    f.write(env_content)

print("Wrote .env with", len(values), "secrets loaded.")

Wrote .env with 9 secrets loaded.


In [3]:
%%writefile db.py
import os, psycopg2
from psycopg2.extras import RealDictCursor
from contextlib import contextmanager
from dotenv import load_dotenv
load_dotenv()

CFG = dict(host=os.getenv("DB_HOST"), port=os.getenv("DB_PORT", "5432"),
           dbname=os.getenv("DB_NAME"), user=os.getenv("DB_USER"),
           password=os.getenv("DB_PASSWORD"), sslmode="require")

@contextmanager
def cursor(commit=False):
    conn = psycopg2.connect(**CFG)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    try:
        yield cur
        if commit: conn.commit()
    finally:
        cur.close(); conn.close()

def init_db():
    with cursor(commit=True) as cur:
        cur.execute("""CREATE TABLE IF NOT EXISTS users (
            id SERIAL PRIMARY KEY, username VARCHAR(50) UNIQUE, email VARCHAR(255) UNIQUE,
            password_hash VARCHAR(255), is_verified BOOLEAN DEFAULT FALSE)""")
        cur.execute("""CREATE TABLE IF NOT EXISTS otp_codes (
            id SERIAL PRIMARY KEY, email VARCHAR(255), code VARCHAR(6),
            purpose VARCHAR(20), expires_at TIMESTAMP, used BOOLEAN DEFAULT FALSE)""")
        cur.execute("""CREATE TABLE IF NOT EXISTS journals (
            id SERIAL PRIMARY KEY, user_id INT, content TEXT,
            detected_mood VARCHAR(50), created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        cur.execute("""CREATE TABLE IF NOT EXISTS mood_logs (
            id SERIAL PRIMARY KEY, user_id INT, mood_score INT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")

        # --- NEW FINAL TABLES ---
        cur.execute("""CREATE TABLE IF NOT EXISTS chat_messages (
            id SERIAL PRIMARY KEY, user_id INT, role VARCHAR(20), content TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        cur.execute("""CREATE TABLE IF NOT EXISTS face_scans (
            id SERIAL PRIMARY KEY, user_id INT, dominant_mood VARCHAR(50),
            confidence FLOAT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")

# --- Existing Functions ---
def save_journal(uid, c, m):
    with cursor(commit=True) as cur: cur.execute("INSERT INTO journals (user_id, content, detected_mood) VALUES (%s, %s, %s)", (uid, c, m))
def get_journals(uid):
    with cursor() as cur: cur.execute("SELECT * FROM journals WHERE user_id=%s ORDER BY created_at DESC LIMIT 10", (uid,)); return cur.fetchall()
def log_mood(uid, s):
    with cursor(commit=True) as cur: cur.execute("INSERT INTO mood_logs (user_id, mood_score) VALUES (%s, %s)", (uid, s))
def get_mood_logs(uid):
    with cursor() as cur: cur.execute("SELECT * FROM mood_logs WHERE user_id=%s ORDER BY created_at ASC", (uid,)); return cur.fetchall()

# --- New AI & Face Functions ---
def save_chat_message(user_id, role, content):
    with cursor(commit=True) as cur:
        cur.execute("INSERT INTO chat_messages (user_id, role, content) VALUES (%s, %s, %s)", (user_id, role, content))

def get_chat_history(user_id):
    with cursor() as cur:
        cur.execute("SELECT role, content FROM chat_messages WHERE user_id=%s ORDER BY id ASC", (user_id,))
        return cur.fetchall()

def save_face_scan(user_id, mood, confidence):
    with cursor(commit=True) as cur:
        cur.execute("INSERT INTO face_scans (user_id, dominant_mood, confidence) VALUES (%s, %s, %s)", (user_id, mood, confidence))

Overwriting db.py


In [4]:
%%writefile auth.py
import os, jwt, bcrypt, random, string
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv
from db import cursor
load_dotenv()

SECRET = os.getenv("JWT_SECRET")

def hash_pw(pw): return bcrypt.hashpw(pw.encode(), bcrypt.gensalt()).decode()
def check_pw(pw, h): return bcrypt.checkpw(pw.encode(), h.encode())

def make_token(user):
    payload = {"id": user["id"], "username": user["username"], "email": user["email"],
               "exp": datetime.now(timezone.utc) + timedelta(hours=1)}
    return jwt.encode(payload, SECRET, algorithm="HS256")

def read_token(token):
    try: return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError: return None

def get_user(email):
    with cursor() as cur:
        cur.execute("SELECT * FROM users WHERE email=%s", (email,))
        return cur.fetchone()

def username_taken(username):
    with cursor() as cur:
        cur.execute("SELECT 1 FROM users WHERE username=%s", (username,))
        return cur.fetchone() is not None

def create_user(username, email, pw):
    with cursor(commit=True) as cur:
        cur.execute("INSERT INTO users (username,email,password_hash) VALUES (%s,%s,%s)",
                    (username, email, hash_pw(pw)))

def verify_user(email):
    with cursor(commit=True) as cur:
        cur.execute("UPDATE users SET is_verified=TRUE WHERE email=%s", (email,))

def set_password(email, pw):
    with cursor(commit=True) as cur:
        cur.execute("UPDATE users SET password_hash=%s WHERE email=%s", (hash_pw(pw), email))

def new_otp():
    return "".join(random.choices(string.digits, k=6))

def save_otp(email, code, purpose):
    exp = datetime.now(timezone.utc) + timedelta(minutes=10)
    with cursor(commit=True) as cur:
        cur.execute("UPDATE otp_codes SET used=TRUE WHERE email=%s AND purpose=%s", (email, purpose))
        cur.execute("INSERT INTO otp_codes (email,code,purpose,expires_at) VALUES (%s,%s,%s,%s)",
                    (email, code, purpose, exp))

def check_otp(email, code, purpose):
    with cursor(commit=True) as cur:
        cur.execute("""SELECT * FROM otp_codes WHERE email=%s AND purpose=%s AND used=FALSE
                       ORDER BY id DESC LIMIT 1""", (email, purpose))
        row = cur.fetchone()
        if not row or row["code"] != code:
            return False
        now = datetime.now(row["expires_at"].tzinfo) if row["expires_at"].tzinfo else datetime.now()
        if now > row["expires_at"]:
            return False
        cur.execute("UPDATE otp_codes SET used=TRUE WHERE id=%s", (row["id"],))
        return True

Overwriting auth.py


In [5]:
%%writefile email_utils.py
import os, smtplib
from email.mime.text import MIMEText
from dotenv import load_dotenv
load_dotenv()

HOST, PORT = "smtp.gmail.com", 587
EMAIL = os.getenv("SMTP_EMAIL")
APP_PW = os.getenv("SMTP_APP_PASSWORD")

def send_otp(to_email, code, purpose):
    subject = "Your Verification Code" if purpose == "signup" else "Your Password Reset Code"
    msg = MIMEText(f"Your code is: {code}\nExpires in 10 minutes.")
    msg["From"], msg["To"], msg["Subject"] = EMAIL, to_email, subject
    try:
        with smtplib.SMTP(HOST, PORT, timeout=15) as s:
            s.starttls()
            s.login(EMAIL, APP_PW)
            s.sendmail(EMAIL, to_email, msg.as_string())
        return True, "sent"
    except Exception as e:
        return False, str(e)

Overwriting email_utils.py


In [6]:
%%writefile app.py
import os, re, requests, tempfile, streamlit as st
from fpdf import FPDF
from db import init_db
from auth import (make_token, read_token, get_user, username_taken, create_user,
                   verify_user, set_password, check_pw, new_otp, save_otp, check_otp)
from email_utils import send_otp

st.set_page_config(page_title="Mood Mentor", page_icon="✨", layout="wide")

st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;800&display=swap');
    header {visibility: hidden;}
    @keyframes gradientBG { 0% { background-position: 0% 50%; } 50% { background-position: 100% 50%; } 100% { background-position: 0% 50%; } }
    @keyframes slideUp { from { opacity: 0; transform: translateY(20px); } to { opacity: 1; transform: translateY(0); } }
    .stApp { background: linear-gradient(-45deg, #89f7fe, #66a6ff, #a18cd1, #fbc2eb); background-size: 400% 400%; animation: gradientBG 12s ease infinite; font-family: 'Inter', sans-serif; }
    .block-container { background: rgba(255, 255, 255, 0.55) !important; backdrop-filter: blur(25px) !important; -webkit-backdrop-filter: blur(25px) !important; border-radius: 32px !important; padding: 3rem 4rem !important; margin-top: 2rem !important; margin-bottom: 2rem !important; box-shadow: 0 20px 50px rgba(0, 0, 0, 0.15), 0 1px 3px rgba(255,255,255,0.5) inset !important; border: 1px solid rgba(255, 255, 255, 0.9) !important; animation: slideUp 0.8s cubic-bezier(0.16, 1, 0.3, 1) forwards; }
    [data-testid="stSidebar"] { background: rgba(255, 255, 255, 0.4) !important; backdrop-filter: blur(20px); border-right: 1px solid rgba(255, 255, 255, 0.6) !important; }
    .stButton>button { background: rgba(255, 255, 255, 0.8) !important; color: #0f172a !important; border-radius: 16px !important; border: 1px solid rgba(255, 255, 255, 1.0) !important; font-weight: 600 !important; padding: 12px 24px !important; transition: all 0.3s cubic-bezier(0.4, 0, 0.2, 1) !important; box-shadow: 0 4px 15px rgba(0,0,0,0.05) !important; width: 100%; backdrop-filter: blur(10px) !important; }
    .stButton>button:hover { transform: translateY(-2px) !important; box-shadow: 0 8px 25px rgba(0,0,0,0.1) !important; background: #ffffff !important; }
    .stTextInput>div>div>input, .stChatInputContainer { background: rgba(255, 255, 255, 0.7) !important; border-radius: 16px !important; border: 1px solid rgba(255, 255, 255, 0.9) !important; padding: 12px !important; transition: all 0.3s ease !important; color: #0f172a !important; }
    .stTextInput>div>div>input:focus, .stChatInputContainer:focus-within { background: rgba(255, 255, 255, 1.0) !important; box-shadow: 0 0 0 4px rgba(255, 255, 255, 0.5) !important; }
    [data-testid="stChatMessage"] { background: rgba(255, 255, 255, 0.7) !important; border-radius: 20px !important; padding: 20px !important; margin-bottom: 16px !important; box-shadow: 0 4px 15px rgba(0, 0, 0, 0.05) !important; border: 1px solid rgba(255, 255, 255, 0.9) !important; animation: slideUp 0.4s ease-out forwards; }
    h1 { font-weight: 800 !important; background: linear-gradient(to right, #0f172a, #475569); -webkit-background-clip: text; -webkit-text-fill-color: transparent; }
    h2, h3, p, span, label { color: #1e293b !important; }
</style>
""", unsafe_allow_html=True)

BACKEND_URL = os.getenv("BACKEND_URL", "http://localhost:8000")

@st.cache_resource
def setup(): init_db()
setup()

if "page" not in st.session_state: st.session_state.page = "login"
if "token" not in st.session_state: st.session_state.token = None
if "chat_history" not in st.session_state: st.session_state.chat_history = []

def goto(p): st.session_state.page = p; st.rerun()
def valid_pw(pw): return len(pw) >= 8 and re.search(r"[A-Za-z]", pw) and re.search(r"[0-9]", pw)

# ---- Logged-in view ----
if st.session_state.token:
    user = read_token(st.session_state.token)
    if user:
        st.title("✨ Employee Wellness Analysis")
        st.caption(f"Logged in securely as {user['username']}")

        col1, col2, col3 = st.columns([1,1,8])
        with col1:
            if st.button("Sign Out"):
                st.session_state.token = None; goto("login")

        st.divider()
        upload_col, chat_col = st.columns([3, 2], gap="large")

        with chat_col:
            st.subheader("💬 Wellness Chat")
            chat_box = st.container(height=350)
            with chat_box:
                for turn in st.session_state.chat_history:
                    with st.chat_message(turn["role"]): st.write(turn["content"])

            user_msg = st.chat_input("How are you feeling today?")
            if user_msg:
                st.session_state.chat_history.append({"role": "user", "content": user_msg})
                recent_history = st.session_state.chat_history[-10:-1]
                try:
                    resp = requests.post(f"{BACKEND_URL}/chat", json={"message": user_msg, "history": recent_history}, headers={"Authorization": f"Bearer {st.session_state.token}"}, timeout=60)
                    if resp.status_code == 200: reply = resp.json()["reply"]
                    else: reply = "Sorry, backend error."
                except: reply = "Sorry, backend is offline."
                st.session_state.chat_history.append({"role": "assistant", "content": reply})
                st.rerun()

            if st.session_state.chat_history and st.button("Clear chat"):
                st.session_state.chat_history = []; st.rerun()

        with upload_col:
            st.subheader("File Upload")
            uploaded = st.file_uploader("Choose a CSV or TXT file", type=["csv", "txt"])
            headers = {"Authorization": f"Bearer {st.session_state.token}"}

            if uploaded is not None:
                is_csv = uploaded.name.lower().endswith(".csv")
                column_name = st.text_input("Feedback column name").strip() or None if is_csv else None

                c1, c2 = st.columns(2)
                if c1.button("Upload & Preview"):
                    files = {"file": (uploaded.name, uploaded.getvalue())}
                    try:
                        resp = requests.post(f"{BACKEND_URL}/upload", files=files, headers=headers, timeout=15)
                        if resp.status_code == 200: st.success(f"Uploaded! {resp.json()['row_count']} rows.")
                        else: st.error("Upload failed.")
                    except: st.error("Backend error.")

                if c2.button("Run NLP Analysis"):
                    files = {"file": (uploaded.name, uploaded.getvalue())}
                    form = {"column": column_name} if column_name else {}
                    with st.spinner("Running multilingual NLP pipeline…"):
                        try:
                            resp = requests.post(f"{BACKEND_URL}/analyze", files=files, data=form, headers=headers, timeout=120)
                            if resp.status_code == 200:
                                r = resp.json()
                                st.subheader("Analysis Report")
                                st.write("**Detected language:**", r["detected_language"])
                                st.bar_chart({"Positive": r["sentiment_scores"]["pos"], "Negative": r["sentiment_scores"]["neg"], "Neutral": r["sentiment_scores"]["neu"]})

                                # NEW: GENERATE PDF BUTTON!
                                st.divider()
                                st.subheader("📄 Export Results")

                                pdf = FPDF()
                                pdf.add_page()
                                pdf.set_font("Arial", size=16, style="B")
                                pdf.cell(200, 10, txt="Employee Wellness Analysis Report", ln=1, align='C')
                                pdf.ln(10)
                                pdf.set_font("Arial", size=12)
                                pdf.cell(200, 10, txt=f"Analyzed File: {r['filename']}", ln=1)
                                pdf.cell(200, 10, txt=f"Language Detected: {r['detected_language']}", ln=1)
                                pdf.cell(200, 10, txt=f"Overall Sentiment: {r['final_sentiment']}", ln=1)
                                pdf.cell(200, 10, txt=f"Dominant Emotion: {r['final_emotion']}", ln=1)
                                pdf.ln(5)
                                pdf.cell(200, 10, txt="Sentiment Breakdown:", ln=1)
                                pdf.cell(200, 10, txt=f"- Positive: {r['sentiment_scores']['pos']}", ln=1)
                                pdf.cell(200, 10, txt=f"- Negative: {r['sentiment_scores']['neg']}", ln=1)
                                pdf.cell(200, 10, txt=f"- Neutral: {r['sentiment_scores']['neu']}", ln=1)

                                with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
                                    pdf.output(tmp.name)
                                    with open(tmp.name, "rb") as f:
                                        pdf_bytes = f.read()

                                st.download_button(label="📥 Download PDF Report", data=pdf_bytes, file_name="MoodMentor_Report.pdf", mime="application/pdf")
                            else: st.error("Analysis failed.")
                        except: st.error("Backend error.")
        st.stop()
    st.session_state.token = None

# ---- Login UI ----
st.title("🔐 Login to Mood Mentor")
if st.session_state.page == "login":
    with st.form("login"):
        email = st.text_input("Email")
        pw = st.text_input("Password", type="password")
        go = st.form_submit_button("Log in")
    if go:
        user = get_user(email.strip().lower())
        if not user or not check_pw(pw, user["password_hash"]): st.error("Invalid email or password.")
        elif not user["is_verified"]: st.session_state.email = user["email"]; goto("verify")
        else: st.session_state.token = make_token(user); st.rerun()

    c1, c2 = st.columns(2)
    if c1.button("Sign up"): goto("signup")
    if c2.button("Forgot password?"): goto("forgot")

elif st.session_state.page == "signup":
    with st.form("signup"):
        username = st.text_input("Username")
        email = st.text_input("Email")
        pw = st.text_input("Password", type="password")
        go = st.form_submit_button("Create account")
    if go:
        email = email.strip().lower()
        if not valid_pw(pw): st.error("Password needs 8+ chars, letters and numbers.")
        else:
            create_user(username, email, pw)
            code = new_otp(); save_otp(email, code, "signup"); send_otp(email, code, "signup")
            st.session_state.email = email; goto("verify")
    if st.button("← Back"): goto("login")
elif st.session_state.page == "verify":
    email = st.session_state.email
    with st.form("verify"):
        code = st.text_input("Code")
        if st.form_submit_button("Verify"):
            if check_otp(email, code.strip(), "signup"): verify_user(email); goto("login")
    if st.button("← Back"): goto("login")

Overwriting app.py


## FastAPI backend (JWT-protected file upload)

This backend exposes a `/upload` endpoint that only accepts `.csv` or `.txt` files, verifies the same JWT issued at Streamlit login, and returns a preview (columns + first rows for CSV, first lines for TXT).


## Multilingual NLP pipeline module

Language detection, text cleaning, Telugu/Kannada stopword filtering, translation to English, lemmatization, VADER sentiment, and keyword-based emotion detection — imported by `backend.py`'s `/analyze` endpoint.


In [7]:
%%writefile nlp_pipeline.py
"""
nlp_pipeline.py
Multilingual NLP pipeline for employee feedback:
normalize -> detect language -> clean -> tokenize -> stopword-filter ->
translate to English -> lemmatize -> sentiment (VADER) -> emotion (Qwen LLM).

Stopword filtering uses the `stopwordsiso` package, which ships stopword
sets for 50+ languages keyed by ISO 639-1 code (the same codes langdetect
returns), so any supported language is handled automatically instead of
needing a hardcoded list per language. If the detected language isn't in
stopwordsiso's coverage, filtering is simply skipped for that text.

Heavy libs (spacy model, translator, vader, Qwen model) load once at import
time via lazy module-level globals, so repeated /analyze calls reuse them.
"""

import re
import json as _json
import ftfy
import emoji
import spacy
import torch
import stopwordsiso
from transformers import AutoModelForCausalLM, AutoTokenizer
from langdetect import detect, DetectorFactory
from deep_translator import GoogleTranslator
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

DetectorFactory.seed = 0

_nlp = None
_vader = None
_qwen_model = None
_qwen_tokenizer = None

QWEN_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Human-readable names for common languages this pipeline is likely to see.
# Purely cosmetic (shown in the UI) -- falls back to the raw ISO code if a
# language isn't listed here, so it never blocks processing of any language.
LANGUAGE_NAMES = {
    "te": "Telugu", "kn": "Kannada", "en": "English", "ta": "Tamil",
    "hi": "Hindi", "ml": "Malayalam", "mr": "Marathi", "bn": "Bengali", "gu": "Gujarati",
    "fr": "French", "de": "German", "es": "Spanish", "pt": "Portuguese",
    "ar": "Arabic", "zh": "Chinese", "ja": "Japanese", "ko": "Korean", "ru": "Russian",
}


def _get_stopwords(language_code: str) -> set:
    """
    Returns the stopword set for `language_code` using stopwordsiso, which
    covers 50+ languages by ISO 639-1 code. Returns an empty set for any
    language it doesn't cover -- filtering is skipped rather than failing,
    so unsupported languages still flow through the rest of the pipeline.
    """
    if stopwordsiso.has_lang(language_code):
        return stopwordsiso.stopwords(language_code)
    return set()

# Fixed label set the Qwen prompt is constrained to. Keeping this list stable
# means downstream code (backend.py, Streamlit charts) never has to change,
# no matter which underlying model produces the emotion.
EMOTION_LABELS = ["Happy", "Sad", "Stress", "Angry", "Fear", "Neutral"]

EMOTION_EMOJI = {
    "Happy": "\U0001F60A", "Sad": "\U0001F622", "Stress": "\U0001F62B",
    "Angry": "\U0001F621", "Fear": "\U0001F628", "Neutral": "\U0001F610",
}


def _get_nlp():
    """Lazy-load the multilingual spaCy model once per process."""
    global _nlp
    if _nlp is None:
        _nlp = spacy.load("xx_sent_ud_sm")
    return _nlp


def _get_vader():
    global _vader
    if _vader is None:
        _vader = SentimentIntensityAnalyzer()
    return _vader


def _get_qwen():
    """Lazy-load Qwen2.5-1.5B-Instruct once per process (GPU if available)."""
    global _qwen_model, _qwen_tokenizer
    if _qwen_model is None:
        _qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)
        _qwen_model = AutoModelForCausalLM.from_pretrained(
            QWEN_MODEL_NAME,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
        )
    return _qwen_model, _qwen_tokenizer


def _qwen_emotion(text: str) -> dict:
    """
    Prompts Qwen to classify `text` into one of EMOTION_LABELS and return a
    confidence score per label as strict JSON. Falls back to Neutral if the
    model output can't be parsed (LLMs don't always obey format instructions).
    """
    model, tokenizer = _get_qwen()

    labels_str = ", ".join(EMOTION_LABELS)
    system_prompt = (
        "You are an emotion classification engine for employee wellness feedback. "
        f"Classify the given text into exactly one of these emotions: {labels_str}. "
        "Respond with ONLY a JSON object, no other text, in this exact format: "
        '{"emotion": "<one label>", "scores": {"Happy": 0-1, "Sad": 0-1, '
        '"Stress": 0-1, "Angry": 0-1, "Fear": 0-1, "Neutral": 0-1}}'
    )
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": text if text.strip() else "(empty feedback)"},
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.1,
            do_sample=False,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    reply = tokenizer.decode(generated, skip_special_tokens=True).strip()

    try:
        # Model may wrap JSON in markdown fences; strip those defensively.
        reply_clean = reply.replace("```json", "").replace("```", "").strip()
        parsed = _json.loads(reply_clean)
        emotion = parsed.get("emotion", "Neutral")
        scores = parsed.get("scores", {})
        if emotion not in EMOTION_LABELS:
            emotion = "Neutral"
        scores = {label: float(scores.get(label, 0.0)) for label in EMOTION_LABELS}
    except Exception:
        emotion, scores = "Neutral", {label: 0.0 for label in EMOTION_LABELS}
        scores["Neutral"] = 1.0

    return {"emotion": emotion, "scores": scores}


def process_employee_feedback(text: str) -> dict:
    """Runs the full pipeline on a single blob of text and returns a results dict."""
    nlp = _get_nlp()
    vader = _get_vader()

    normalized_text = ftfy.fix_text(text)

    try:
        language = detect(normalized_text)
    except Exception:
        language = "unknown"
    detected_language = LANGUAGE_NAMES.get(language, "Other / Unknown")

    emoji_list = [ch for ch in normalized_text if ch in emoji.EMOJI_DATA]

    cleaned_text = re.sub(r"https?://\S+|www\.\S+", " ", normalized_text)
    cleaned_text = re.sub(r"\S+@\S+", " ", cleaned_text)
    cleaned_text = re.sub(r"@\w+|#\w+", " ", cleaned_text)
    cleaned_text = emoji.replace_emoji(cleaned_text, replace="")
    cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

    doc = nlp(cleaned_text)
    sentences = [s.text.strip() for s in doc.sents if s.text.strip()]
    original_tokens = [t.text for t in doc if not t.is_space]
    clean_tokens = [t.text for t in doc if not t.is_punct and not t.is_space and not t.like_num]

    selected_stopwords = _get_stopwords(language)
    filtered_tokens = [t for t in clean_tokens if t.lower() not in selected_stopwords]
    final_preprocessed_text = " ".join(filtered_tokens)

    try:
        translated_text = GoogleTranslator(source="auto", target="en").translate(final_preprocessed_text)
    except Exception as error:
        translated_text = f"Translation failed: {error}"

    english_doc = nlp(translated_text)
    lemmas = [t.lemma_ if t.lemma_ else t.text for t in english_doc if not t.is_space]
    lemmatized_text = " ".join(lemmas)

    sentiment_scores = vader.polarity_scores(translated_text)
    compound_score = sentiment_scores["compound"]
    if compound_score >= 0.05:
        final_sentiment = "Positive \U0001F60A"
    elif compound_score <= -0.05:
        final_sentiment = "Negative \U0001F614"
    else:
        final_sentiment = "Neutral \U0001F610"

    # --- Emotion detection via Qwen LLM (replaces old keyword matching) ---
    qwen_result = _qwen_emotion(translated_text)
    emotion_scores = qwen_result["scores"]
    final_emotion_label = qwen_result["emotion"]
    final_emotion = f"{final_emotion_label} {EMOTION_EMOJI.get(final_emotion_label, '')}"

    return {
        "language_code": language,
        "detected_language": detected_language,
        "normalized_text": normalized_text,
        "cleaned_text": cleaned_text,
        "sentences": sentences,
        "original_tokens": original_tokens,
        "filtered_tokens": filtered_tokens,
        "emoji_list": emoji_list,
        "final_preprocessed_text": final_preprocessed_text,
        "translated_text": translated_text,
        "lemmatized_text": lemmatized_text,
        "sentiment_scores": sentiment_scores,
        "final_sentiment": final_sentiment,
        "emotion_scores": emotion_scores,
        "final_emotion": final_emotion,
    }


# --- Crisis-keyword safety net for the wellness chatbot -----------------
# This is a simple, deliberately blunt keyword check that runs regardless
# of what the LLM says. If it fires, we always show crisis resources —
# we never rely on the small LLM alone to catch something this important.
CRISIS_KEYWORDS = [
    "suicide", "kill myself", "end my life", "want to die", "self harm",
    "self-harm", "hurt myself", "not worth living", "no reason to live",
]

CRISIS_MESSAGE = (
    "I'm really glad you reached out, and I want to make sure you get support "
    "beyond what I can offer here. If you're in immediate danger, please contact "
    "your local emergency number right now. You can also reach a crisis line: "
    "in India, AASRA is available at +91-9820466726 (24/7). If you're outside "
    "India, please look up a local crisis helpline or talk to a trusted person "
    "or your HR/EAP contact. You don't have to go through this alone."
)

WELLNESS_SYSTEM_PROMPT = (
    "You are a supportive workplace wellness assistant for employees. "
    "Your role is to listen, validate feelings, and offer general, gentle "
    "coping suggestions (like breathing exercises, taking a short break, "
    "or talking to a trusted colleague or manager). "
    "You are NOT a therapist or doctor: never diagnose any condition, never "
    "claim expertise you don't have, and never give medical or medication "
    "advice. If the employee describes something serious (ongoing crisis, "
    "self-harm, harming others), gently encourage them to contact a mental "
    "health professional, their HR/EAP program, or a crisis helpline. "
    "Keep replies short (2-4 sentences), warm, and non-judgmental. "
    "Avoid clinical labels and avoid being preachy or repetitive."
)


def _contains_crisis_language(text: str) -> bool:
    lowered = text.lower()
    return any(kw in lowered for kw in CRISIS_KEYWORDS)


def wellness_chat_reply(message: str, history: list[dict] | None = None) -> dict:
    """
    Generates a supportive wellness chatbot reply using the same Qwen model
    already loaded for emotion detection.

    `history` is an optional list of {"role": "user"|"assistant", "content": str}
    dicts representing prior turns in the conversation (kept short/recent by
    the caller — this function does not trim it).

    Always checks for crisis language first; if found, returns a fixed,
    resource-pointing message instead of an LLM-generated one, since we
    never want a small model improvising in a safety-critical moment.
    """
    if _contains_crisis_language(message):
        return {"reply": CRISIS_MESSAGE, "flagged": True}

    model, tokenizer = _get_qwen()

    messages = [{"role": "system", "content": WELLNESS_SYSTEM_PROMPT}]
    for turn in (history or []):
        if turn.get("role") in ("user", "assistant") and turn.get("content"):
            messages.append({"role": turn["role"], "content": turn["content"]})
    messages.append({"role": "user", "content": message})

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    reply = tokenizer.decode(generated, skip_special_tokens=True).strip()

    if not reply:
        reply = "I'm here and listening — could you tell me a bit more about how you're feeling?"

    return {"reply": reply, "flagged": False}


Overwriting nlp_pipeline.py


In [8]:
%%writefile backend.py
import os, io, jwt, csv
from fastapi import FastAPI, UploadFile, File, Form, Header, HTTPException
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
from dotenv import load_dotenv
from nlp_pipeline import process_employee_feedback, wellness_chat_reply
load_dotenv()

SECRET = os.getenv("JWT_SECRET")
app = FastAPI(title="Upload API")

app.add_middleware(CORSMiddleware, allow_origins=["*"],
                    allow_methods=["*"], allow_headers=["*"])

def get_user(authorization: str = Header(None)):
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(401, "Missing token")
    token = authorization.split(" ", 1)[1]
    try:
        return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError:
        raise HTTPException(401, "Invalid or expired token")

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/upload")
async def upload(file: UploadFile = File(...), authorization: str = Header(None)):
    user = get_user(authorization)

    name = file.filename or ""
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    if ext not in ("csv", "txt"):
        raise HTTPException(400, "Only .csv or .txt files are allowed.")

    raw = await file.read()
    max_bytes = 5 * 1024 * 1024  # 5 MB cap
    if len(raw) > max_bytes:
        raise HTTPException(400, "File too large (max 5 MB).")

    try:
        text = raw.decode("utf-8")
    except UnicodeDecodeError:
        raise HTTPException(400, "File must be UTF-8 text.")

    lines = text.splitlines()
    row_count = len(lines)
    preview_lines = lines[:20]

    columns = None
    preview_rows = None
    if ext == "csv":
        reader = csv.reader(io.StringIO(text))
        rows = list(reader)
        if rows:
            columns = rows[0]
            preview_rows = rows[1:21]
            row_count = max(len(rows) - 1, 0)  # exclude header from data row count

    return {
        "filename": name,
        "type": ext,
        "uploaded_by": user["username"],
        "row_count": row_count,
        "columns": columns,
        "preview_rows": preview_rows,
        "preview_lines": None if ext == "csv" else preview_lines,
    }


def _extract_text_blob(raw: bytes, ext: str, column: str | None) -> tuple[str, str | None]:
    """
    Returns (text_blob, used_column). For TXT, used_column is None.
    For CSV, joins all non-empty values of the chosen column (or the last
    column if none/invalid was specified) into one whitespace-joined blob —
    matches the notebook's "whole file as one blob" behavior.
    """
    text = raw.decode("utf-8")

    if ext == "txt":
        return text.strip(), None

    reader = csv.reader(io.StringIO(text))
    rows = list(reader)
    if not rows:
        raise HTTPException(400, "CSV file has no rows.")

    header = rows[0]
    data_rows = rows[1:]
    if not data_rows:
        raise HTTPException(400, "CSV file has a header but no data rows.")

    col_index = None
    if column and column in header:
        col_index = header.index(column)
    else:
        col_index = len(header) - 1  # default to last column

    values = [row[col_index] for row in data_rows if len(row) > col_index and row[col_index].strip()]
    blob = " ".join(values).strip()
    if not blob:
        raise HTTPException(400, f"Column '{header[col_index]}' has no readable text.")
    return blob, header[col_index]


@app.post("/analyze")
async def analyze(file: UploadFile = File(...), column: str = Form(None),
                   authorization: str = Header(None)):
    """
    Runs the multilingual NLP pipeline (language detection, cleaning,
    stopword filtering, translation, lemmatization, VADER sentiment,
    keyword-based emotion) on an uploaded .csv or .txt file.
    """
    get_user(authorization)  # just verifies the token; raises 401 if invalid

    name = file.filename or ""
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    if ext not in ("csv", "txt"):
        raise HTTPException(400, "Only .csv or .txt files are allowed.")

    raw = await file.read()
    max_bytes = 5 * 1024 * 1024
    if len(raw) > max_bytes:
        raise HTTPException(400, "File too large (max 5 MB).")

    try:
        text_blob, used_column = _extract_text_blob(raw, ext, column)
    except UnicodeDecodeError:
        raise HTTPException(400, "File must be UTF-8 text.")

    results = process_employee_feedback(text_blob)
    results["filename"] = name
    results["file_type"] = ext.upper()
    results["used_column"] = used_column
    results["original_char_count"] = len(text_blob)
    return results


class ChatTurn(BaseModel):
    role: str
    content: str


class ChatRequest(BaseModel):
    message: str
    history: list[ChatTurn] = []


@app.post("/chat")
async def chat(payload: ChatRequest, authorization: str = Header(None)):
    """
    Wellness support chatbot endpoint. Stateless on the server: the client
    (Streamlit) sends the recent conversation history along with each new
    message, and we generate the next reply with the same Qwen model used
    for emotion detection.
    """
    get_user(authorization)  # verifies the token; raises 401 if invalid

    message = payload.message.strip()
    if not message:
        raise HTTPException(400, "Message cannot be empty.")

    history = [turn.dict() for turn in payload.history]
    result = wellness_chat_reply(message, history=history)
    return result


Overwriting backend.py


In [9]:
from db import init_db
init_db()
print("✅ Connected to PostgreSQL and ensured tables exist.")

✅ Connected to PostgreSQL and ensured tables exist.


In [10]:
!mkdir -p .streamlit
!mkdir -p pages
!pip install -q opencv-python-headless
!pip install -q deepface google-generativeai python-dotenv
!pip install -q plotly fpdf2
!pip install -q mtcnn

# Fetch the Gemini key from Colab secrets and append it to our backend's .env file
from google.colab import userdata
try:
    gemini_key = userdata.get('GEMINI_API_KEY')
    with open(".env", "a") as f:
        f.write(f"\nGEMINI_API_KEY={gemini_key}\n")
    print("✅ Successfully linked Gemini API Key from Colab Secrets!")
except Exception as e:
    print("⚠️ Could not find 'GEMINI_API_KEY' in Colab secrets. Please click the Key icon on the left to add it.")

✅ Successfully linked Gemini API Key from Colab Secrets!


In [11]:
%%writefile .streamlit/config.toml
[theme]
base="light"
primaryColor="#007AFF"
backgroundColor="#FFFFFF"
secondaryBackgroundColor="#F2F2F7"
textColor="#1C1C1E"
font="sans serif"

Overwriting .streamlit/config.toml


In [12]:
%%writefile pages/1_Profile_Dashboard.py
import streamlit as st, os, pandas as pd
import plotly.express as px
from auth import read_token
from db import get_mood_logs

st.set_page_config(page_title="Profile | Mood Mentor", page_icon="👤", layout="wide")

st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;800&display=swap');
    header {visibility: hidden;}
    @keyframes gradientBG { 0% { background-position: 0% 50%; } 50% { background-position: 100% 50%; } 100% { background-position: 0% 50%; } }
    @keyframes slideUp { from { opacity: 0; transform: translateY(20px); } to { opacity: 1; transform: translateY(0); } }
    .stApp { background: linear-gradient(-45deg, #89f7fe, #66a6ff, #a18cd1, #fbc2eb); background-size: 400% 400%; animation: gradientBG 12s ease infinite; font-family: 'Inter', sans-serif; }
    .block-container { background: rgba(255, 255, 255, 0.55) !important; backdrop-filter: blur(25px) !important; -webkit-backdrop-filter: blur(25px) !important; border-radius: 32px !important; padding: 3rem 4rem !important; margin-top: 2rem !important; margin-bottom: 2rem !important; box-shadow: 0 20px 50px rgba(0, 0, 0, 0.15), 0 1px 3px rgba(255,255,255,0.5) inset !important; border: 1px solid rgba(255, 255, 255, 0.9) !important; animation: slideUp 0.8s cubic-bezier(0.16, 1, 0.3, 1) forwards; }
    [data-testid="stSidebar"] { background: rgba(255, 255, 255, 0.4) !important; backdrop-filter: blur(20px); border-right: 1px solid rgba(255, 255, 255, 0.6) !important; }
    h1 { font-weight: 800 !important; background: linear-gradient(to right, #0f172a, #475569); -webkit-background-clip: text; -webkit-text-fill-color: transparent; }
    h2, h3, p, span, label { color: #1e293b !important; }
    .profile-card { background: rgba(255, 255, 255, 0.6); padding: 40px; border-radius: 24px; border: 1px solid rgba(255, 255, 255, 0.9); margin-bottom: 20px; text-align: center; box-shadow: 0 4px 15px rgba(0,0,0,0.05); backdrop-filter: blur(10px);}
    .profile-card img { border-radius: 50%; box-shadow: 0 10px 25px rgba(0,0,0,0.1); margin-bottom: 20px; }
    .stat-card { background: rgba(255, 255, 255, 0.7); padding: 25px; border-radius: 20px; text-align: center; border: 1px solid rgba(255, 255, 255, 0.9); box-shadow: 0 4px 15px rgba(0,0,0,0.05); transition: transform 0.3s ease; backdrop-filter: blur(10px);}
    .stat-card:hover { transform: translateY(-5px); }
    .stat-card h3 { color: #007AFF !important; margin-bottom: 5px; font-size: 2.5rem; font-weight: 800;}
</style>
""", unsafe_allow_html=True)

if "token" not in st.session_state or not st.session_state.token:
    st.warning("Please log in from the main app.")
    st.stop()

user = read_token(st.session_state.token)
st.title("👤 Profile Dashboard")
st.markdown("Welcome to **Mood Mentor** - Your personal emotional wellness companion.")
st.divider()

# Fetch real mood logs from database
mood_data = get_mood_logs(user["id"])
avg_mood = sum([m['mood_score'] for m in mood_data]) // len(mood_data) if mood_data else 85

col1, col2 = st.columns([1, 2], gap="large")
with col1:
    st.markdown('<div class="profile-card">', unsafe_allow_html=True)
    if os.path.exists("profile.jpg"): st.image("profile.jpg", width=180)
    else: st.image("https://cdn-icons-png.flaticon.com/512/3135/3135715.png", width=180)
    st.markdown("### Anil Kumar Soma")
    st.markdown("📧 somaanil14@gmail.com")
    st.markdown("🎓 **Role:** Student, AIML")
    st.markdown("🟢 **Active & Ready**")
    st.markdown('</div>', unsafe_allow_html=True)
with col2:
    st.markdown("### 📊 Wellness Overview")
    sc1, sc2, sc3 = st.columns(3)
    with sc1: st.markdown(f'<div class="stat-card"><h3>{avg_mood}%</h3><p>Overall Mood</p></div>', unsafe_allow_html=True)
    with sc2: st.markdown(f'<div class="stat-card"><h3>{len(mood_data)}</h3><p>Journals Logged</p></div>', unsafe_allow_html=True)
    with sc3: st.markdown('<div class="stat-card"><h3>34</h3><p>Days Active</p></div>', unsafe_allow_html=True)

    st.markdown("<br>### 📈 Mood Trend Analytics", unsafe_allow_html=True)

    if not mood_data:
        st.info("Write some journals in the AI Journal page to see your mood trend here!")
    else:
        # Create an interactive Plotly chart!
        df = pd.DataFrame(mood_data)
        df['created_at'] = pd.to_datetime(df['created_at'])
        df = df.sort_values('created_at')

        fig = px.area(df, x='created_at', y='mood_score',
                      title="Your Wellness Progress Over Time",
                      labels={'created_at': 'Date', 'mood_score': 'Wellness Score (0-100)'},
                      color_discrete_sequence=['#007AFF'])

        fig.update_layout(plot_bgcolor="rgba(255,255,255,0.4)", paper_bgcolor="rgba(0,0,0,0)",
                          margin=dict(l=20, r=20, t=40, b=20))
        st.plotly_chart(fig, use_container_width=True)

Overwriting pages/1_Profile_Dashboard.py


In [13]:
%%writefile pages/2_Face_Detection.py
import streamlit as st, cv2, numpy as np, tempfile, os
from auth import read_token
from db import log_mood, save_face_scan

try: from deepface import DeepFace; DEEPFACE_READY = True
except: DEEPFACE_READY = False
st.set_page_config(page_title="Mood Analysis", page_icon="🧠", layout="wide")

st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;800&display=swap');
    header {visibility: hidden;}
    @keyframes gradientBG { 0% { background-position: 0% 50%; } 50% { background-position: 100% 50%; } 100% { background-position: 0% 50%; } }
    @keyframes slideUp { from { opacity: 0; transform: translateY(20px); } to { opacity: 1; transform: translateY(0); } }
    .stApp { background: linear-gradient(-45deg, #89f7fe, #66a6ff, #a18cd1, #fbc2eb); background-size: 400% 400%; animation: gradientBG 12s ease infinite; font-family: 'Inter', sans-serif; }
    .block-container { background: rgba(255, 255, 255, 0.55) !important; backdrop-filter: blur(25px) !important; border-radius: 32px !important; padding: 3rem 4rem !important; margin-top: 2rem !important; margin-bottom: 2rem !important; box-shadow: 0 20px 50px rgba(0, 0, 0, 0.15); border: 1px solid rgba(255, 255, 255, 0.9) !important; animation: slideUp 0.8s cubic-bezier(0.16, 1, 0.3, 1) forwards; }
    [data-testid="stSidebar"] { background: rgba(255, 255, 255, 0.4) !important; backdrop-filter: blur(20px); border-right: 1px solid rgba(255, 255, 255, 0.6) !important; }
    h1 { font-weight: 800 !important; background: linear-gradient(to right, #0f172a, #475569); -webkit-background-clip: text; -webkit-text-fill-color: transparent; }
    h2, h3, p, span, label { color: #1e293b !important; }
    .stButton>button { background: rgba(255, 255, 255, 0.8) !important; color: #0f172a !important; border-radius: 16px !important; font-weight: 600 !important; width: 100%; transition: all 0.3s ease; }
</style>
""", unsafe_allow_html=True)

if "token" not in st.session_state or not st.session_state.token:
    st.warning("Please log in first from the main app.")
    st.stop()

user = read_token(st.session_state.token)

st.title("🧠 Advanced Biometric Face Analysis")
if not DEEPFACE_READY: st.error("DeepFace library is initializing.")

tab1, tab2 = st.tabs(["📷 Live Camera Scan", "📁 Upload Image"])

def analyze_and_display(image_bytes):
    img_array = np.asarray(bytearray(image_bytes), dtype=np.uint8)
    image = cv2.imdecode(img_array, 1)

    col1, col2 = st.columns([1,1], gap="large")

    with col2:
        with st.spinner("Executing Deep Learning MTCNN Scan..."):
            with tempfile.NamedTemporaryFile(delete=False, suffix='.jpg') as tmp_file:
                cv2.imwrite(tmp_file.name, image); tmp_path = tmp_file.name
            try:
                # Upgraded to MTCNN backend
                results = DeepFace.analyze(tmp_path, actions=['emotion'], enforce_detection=False, detector_backend='mtcnn')
                if not isinstance(results, list): results = [results]

                # Draw High-Tech Targeting Boxes!
                for face in results:
                    x, y, w, h = face['region']['x'], face['region']['y'], face['region']['w'], face['region']['h']
                    dom_emotion = face['dominant_emotion'].capitalize()
                    conf = face['emotion'][face['dominant_emotion']]

                    # Draw Blue Bounding Box
                    cv2.rectangle(image, (x, y), (x+w, y+h), (255, 150, 0), 3)

                    # Draw High-Tech Label Background
                    cv2.rectangle(image, (x, y-35), (x+w, y), (255, 150, 0), -1)

                    # Put Label Text
                    cv2.putText(image, f"{dom_emotion} ({conf:.0f}%)", (x+5, y-8), cv2.FONT_HERSHEY_DUPLEX, 0.7, (255, 255, 255), 1)

                with col1:
                    st.image(image, channels="BGR", caption="Processed MTCNN Scan", use_column_width=True)

                st.success(f"Biometric Scan Complete! {len(results)} face(s) mapped.")

                # Database Logging
                for i, face_data in enumerate(results):
                    dom = face_data['dominant_emotion'].capitalize()
                    happy = float(face_data['emotion']['happy'])
                    sad = float(face_data['emotion']['sad'])
                    stress = float(face_data['emotion']['fear']) + float(face_data['emotion']['angry'])

                    if happy > 40: score = int(50 + (happy/2))
                    elif sad > 30 or stress > 30: score = int(50 - (sad/2) - (stress/2))
                    else: score = 50

                    # FIXED: Wrap the AI confidence score in standard python float()
                    conf_score = float(face_data['emotion'][face_data['dominant_emotion']])

                    save_face_scan(user["id"], dom, conf_score)
                    log_mood(user["id"], score)

                    st.markdown(f"### Face {i+1} Telemetry")
                    st.markdown(f"**Dominant State:** `{dom}`")
                    st.markdown(f"**Biometric Wellness Score:** `{score}/100` (Saved to DB)")

                    for emotion, score_val in sorted(face_data['emotion'].items(), key=lambda item: item[1], reverse=True)[:3]:
                        st.progress(int(score_val)/100, text=f"{emotion.capitalize()}: {score_val:.1f}%")

            except Exception as e: st.error(f"Error during scan: {e}")
            finally: os.remove(tmp_path)

with tab1:
    camera_photo = st.camera_input("Initiate Biometric Camera Scan")
    if camera_photo and DEEPFACE_READY: analyze_and_display(camera_photo.read())
with tab2:
    uploaded_image = st.file_uploader("Upload Image for Scanning", type=["jpg", "jpeg", "png"])
    if uploaded_image and DEEPFACE_READY: analyze_and_display(uploaded_image.read())

Overwriting pages/2_Face_Detection.py


In [14]:
%%writefile pages/3_AI_Assistant.py
import streamlit as st, os, requests
from dotenv import load_dotenv
from auth import read_token
from db import get_chat_history, save_chat_message
load_dotenv()
st.set_page_config(page_title="AI Mentor", page_icon="🤖", layout="wide")

st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;800&display=swap');
    header {visibility: hidden;}
    @keyframes gradientBG { 0% { background-position: 0% 50%; } 50% { background-position: 100% 50%; } 100% { background-position: 0% 50%; } }
    @keyframes slideUp { from { opacity: 0; transform: translateY(20px); } to { opacity: 1; transform: translateY(0); } }
    .stApp { background: linear-gradient(-45deg, #89f7fe, #66a6ff, #a18cd1, #fbc2eb); background-size: 400% 400%; animation: gradientBG 12s ease infinite; font-family: 'Inter', sans-serif; }
    .block-container { background: rgba(255, 255, 255, 0.55) !important; backdrop-filter: blur(25px) !important; border-radius: 32px !important; padding: 3rem 4rem !important; margin-top: 2rem !important; margin-bottom: 2rem !important; box-shadow: 0 20px 50px rgba(0,0,0,0.15), 0 1px 3px rgba(255,255,255,0.5) inset !important; border: 1px solid rgba(255,255,255,0.9) !important; animation: slideUp 0.8s cubic-bezier(0.16, 1, 0.3, 1) forwards; }
    [data-testid="stSidebar"] { background: rgba(255, 255, 255, 0.4) !important; backdrop-filter: blur(20px); border-right: 1px solid rgba(255, 255, 255, 0.6) !important; }
    .stTextInput>div>div>input, .stChatInputContainer { background: rgba(255, 255, 255, 0.7) !important; border-radius: 16px !important; border: 1px solid rgba(255,255,255,0.9) !important; padding: 12px !important; color: #0f172a !important; }
    [data-testid="stChatMessage"] { background: rgba(255, 255, 255, 0.7) !important; border-radius: 20px !important; padding: 20px !important; margin-bottom: 16px !important; box-shadow: 0 4px 15px rgba(0,0,0,0.05) !important; border: 1px solid rgba(255,255,255,0.9) !important; animation: slideUp 0.4s ease-out forwards; }
    h1 { font-weight: 800 !important; background: linear-gradient(to right, #0f172a, #475569); -webkit-background-clip: text; -webkit-text-fill-color: transparent; }
    h2, h3, p, span, label { color: #1e293b !important; }
</style>
""", unsafe_allow_html=True)

if "token" not in st.session_state or not st.session_state.token:
    st.warning("Please log in first from the main app.")
    st.stop()

user = read_token(st.session_state.token)

st.title("🤖 AI Mood Mentor")
st.markdown("Your entire conversation history is securely backed up in our Postgres Database.")

# Fetch history directly from DB!
db_history = get_chat_history(user["id"])
if not db_history:
    st.session_state.gemini_chat_history = [{"role": "assistant", "content": f"Hello {user['username']}! I am Mood Mentor. How can I help you today?"}]
else:
    st.session_state.gemini_chat_history = db_history

for msg in st.session_state.gemini_chat_history:
    with st.chat_message(msg["role"]): st.markdown(msg["content"])

user_input = st.chat_input("Ask Mood Mentor for advice...")
if user_input:
    # 1. Show user message and SAVE TO DB
    st.session_state.gemini_chat_history.append({"role": "user", "content": user_input})
    save_chat_message(user["id"], "user", user_input)
    with st.chat_message("user"): st.markdown(user_input)

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            bot_reply = None
            GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
            if GEMINI_API_KEY:
                contents = [{"role": "user" if m["role"]=="user" else "model", "parts": [{"text": m["content"]}]} for m in st.session_state.gemini_chat_history[-10:]]
                for model_name in ["gemini-1.5-flash", "gemini-1.5-pro"]:
                    try:
                        resp = requests.post(f"https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent?key={GEMINI_API_KEY}", json={"contents": contents}, timeout=10)
                        if resp.status_code == 200: bot_reply = resp.json()["candidates"][0]["content"]["parts"][0]["text"]; break
                    except: pass
            if not bot_reply: bot_reply = "I'm experiencing a bit of network traffic, but I hear you! Take a deep breath."

            # 2. Show AI reply and SAVE TO DB
            st.markdown(bot_reply)
            st.session_state.gemini_chat_history.append({"role": "assistant", "content": bot_reply})
            save_chat_message(user["id"], "assistant", bot_reply)

Overwriting pages/3_AI_Assistant.py


In [15]:
%%writefile pages/4_Journal.py
import streamlit as st, os
from auth import read_token
from db import save_journal, get_journals, log_mood
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

st.set_page_config(page_title="Journal | Mood Mentor", page_icon="📓", layout="wide")

st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;800&display=swap');
    header {visibility: hidden;}
    @keyframes gradientBG { 0% { background-position: 0% 50%; } 50% { background-position: 100% 50%; } 100% { background-position: 0% 50%; } }
    @keyframes slideUp { from { opacity: 0; transform: translateY(20px); } to { opacity: 1; transform: translateY(0); } }
    .stApp { background: linear-gradient(-45deg, #89f7fe, #66a6ff, #a18cd1, #fbc2eb); background-size: 400% 400%; animation: gradientBG 12s ease infinite; font-family: 'Inter', sans-serif; }
    .block-container { background: rgba(255, 255, 255, 0.55) !important; backdrop-filter: blur(25px) !important; -webkit-backdrop-filter: blur(25px) !important; border-radius: 32px !important; padding: 3rem 4rem !important; margin-top: 2rem !important; margin-bottom: 2rem !important; box-shadow: 0 20px 50px rgba(0, 0, 0, 0.15), 0 1px 3px rgba(255,255,255,0.5) inset !important; border: 1px solid rgba(255, 255, 255, 0.9) !important; animation: slideUp 0.8s cubic-bezier(0.16, 1, 0.3, 1) forwards; }
    [data-testid="stSidebar"] { background: rgba(255, 255, 255, 0.4) !important; backdrop-filter: blur(20px); border-right: 1px solid rgba(255, 255, 255, 0.6) !important; }
    h1 { font-weight: 800 !important; background: linear-gradient(to right, #0f172a, #475569); -webkit-background-clip: text; -webkit-text-fill-color: transparent; }
    h2, h3, p, span, label { color: #1e293b !important; }
    .stTextArea>div>div>textarea { background: rgba(255, 255, 255, 0.7) !important; border-radius: 16px !important; border: 1px solid rgba(255, 255, 255, 0.9) !important; padding: 15px !important; color: #0f172a !important; font-size: 1.1rem; }
    .stButton>button { background: rgba(255, 255, 255, 0.8) !important; color: #0f172a !important; border-radius: 16px !important; border: 1px solid rgba(255, 255, 255, 1.0) !important; font-weight: 600 !important; padding: 12px 24px !important; box-shadow: 0 4px 15px rgba(0,0,0,0.05) !important; width: 100%; }
    .stButton>button:hover { transform: translateY(-2px) !important; box-shadow: 0 8px 25px rgba(0,0,0,0.1) !important; background: #ffffff !important; }
    .journal-card { background: rgba(255,255,255,0.6); padding: 20px; border-radius: 16px; border: 1px solid rgba(255,255,255,0.9); margin-bottom: 15px; box-shadow: 0 4px 10px rgba(0,0,0,0.03); }
</style>
""", unsafe_allow_html=True)

if "token" not in st.session_state or not st.session_state.token:
    st.warning("Please log in first from the main app.")
    st.stop()

user = read_token(st.session_state.token)
if not user:
    st.error("Invalid session. Please login again.")
    st.stop()

st.title("📓 AI Gratitude Journal")
st.markdown("Write down your thoughts. The AI will analyze your true sentiment using VADER NLP.")

entry = st.text_area("What are you grateful for today? Or what's on your mind?", height=150)

if st.button("Save & Analyze Entry"):
    if len(entry.strip()) < 5:
        st.error("Please write a bit more!")
    else:
        with st.spinner("Analyzing your thoughts with NLP..."):
            # Advanced VADER NLP Analysis
            analyzer = SentimentIntensityAnalyzer()
            vs = analyzer.polarity_scores(entry)
            compound = vs['compound']  # Score from -1.0 to 1.0

            # Map NLP score to a 0-100 Mood Score
            score = int(((compound + 1) / 2) * 100)

            if compound >= 0.05: mood = "Happy"
            elif compound <= -0.05:
                mood = "Stressed" if any(w in entry.lower() for w in ['stress', 'anxious', 'worry', 'overwhelmed', 'work']) else "Sad"
            else: mood = "Neutral"

            save_journal(user["id"], entry, mood)
            log_mood(user["id"], score)

            st.success(f"Entry saved! AI detected sentiment score: {score}/100 (**{mood}**)")
            st.balloons()

st.divider()
st.subheader("🕰️ Past Entries")
journals = get_journals(user["id"])

if not journals:
    st.info("No journal entries yet! Write your first one above.")
else:
    for j in journals:
        date_str = j['created_at'].strftime("%B %d, %Y - %H:%M") if hasattr(j['created_at'], 'strftime') else str(j['created_at'])
        st.markdown(f"""
        <div class="journal-card">
            <small style="color:#64748b;">{date_str} | Detected Mood: <b>{j['detected_mood']}</b></small>
            <p style="margin-top: 10px; font-size: 1.05rem;">"{j['content']}"</p>
        </div>
        """, unsafe_allow_html=True)

Overwriting pages/4_Journal.py


In [16]:
%%writefile pages/5_Relax.py
import streamlit as st, os
from auth import read_token
from db import get_journals

st.set_page_config(page_title="Relax | Mood Mentor", page_icon="🧘", layout="wide")

st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;800&display=swap');
    header {visibility: hidden;}
    @keyframes gradientBG { 0% { background-position: 0% 50%; } 50% { background-position: 100% 50%; } 100% { background-position: 0% 50%; } }
    @keyframes slideUp { from { opacity: 0; transform: translateY(20px); } to { opacity: 1; transform: translateY(0); } }
    .stApp { background: linear-gradient(-45deg, #89f7fe, #66a6ff, #a18cd1, #fbc2eb); background-size: 400% 400%; animation: gradientBG 12s ease infinite; font-family: 'Inter', sans-serif; }
    .block-container { background: rgba(255, 255, 255, 0.55) !important; backdrop-filter: blur(25px) !important; -webkit-backdrop-filter: blur(25px) !important; border-radius: 32px !important; padding: 3rem 4rem !important; margin-top: 2rem !important; margin-bottom: 2rem !important; box-shadow: 0 20px 50px rgba(0, 0, 0, 0.15), 0 1px 3px rgba(255,255,255,0.5) inset !important; border: 1px solid rgba(255, 255, 255, 0.9) !important; animation: slideUp 0.8s cubic-bezier(0.16, 1, 0.3, 1) forwards; }
    [data-testid="stSidebar"] { background: rgba(255, 255, 255, 0.4) !important; backdrop-filter: blur(20px); border-right: 1px solid rgba(255, 255, 255, 0.6) !important; }
    h1 { font-weight: 800 !important; background: linear-gradient(to right, #0f172a, #475569); -webkit-background-clip: text; -webkit-text-fill-color: transparent; }
    h2, h3, p, span, label { color: #1e293b !important; }

    /* Beautiful Breathing Animation */
    @keyframes breathe {
        0% { transform: scale(0.8); opacity: 0.7; box-shadow: 0 0 0 0 rgba(255,255,255,0.7); }
        50% { transform: scale(1.4); opacity: 1; box-shadow: 0 0 0 25px rgba(255,255,255,0); }
        100% { transform: scale(0.8); opacity: 0.7; box-shadow: 0 0 0 0 rgba(255,255,255,0); }
    }
    .breathing-circle {
        width: 160px; height: 160px;
        background: linear-gradient(135deg, #a1c4fd, #c2e9fb);
        border-radius: 50%;
        margin: 50px auto;
        animation: breathe 8s infinite ease-in-out;
        display: flex; align-items: center; justify-content: center;
        color: white; font-weight: bold; font-size: 1.3rem;
        box-shadow: 0 10px 30px rgba(0,0,0,0.1);
        border: 2px solid white;
    }
</style>
""", unsafe_allow_html=True)

if "token" not in st.session_state or not st.session_state.token:
    st.warning("Please log in first from the main app.")
    st.stop()

user = read_token(st.session_state.token)
st.title("🧘 Smart Music Therapy & Breathing")
st.markdown("Take a moment for yourself. This page dynamically adjusts based on your recent mood entries.")

journals = get_journals(user["id"])
latest_mood = journals[0]['detected_mood'] if journals else "Neutral"

col1, col2 = st.columns([1, 1], gap="large")

with col1:
    st.subheader("🌬️ Guided Breathing")
    st.markdown("Follow the circle. Breathe in as it expands, hold, and breathe out as it shrinks.")
    st.markdown('<div class="breathing-circle">Breathe</div>', unsafe_allow_html=True)
    st.info("💡 **Tip:** This 4-7-8 breathing technique reduces anxiety in just 60 seconds.")

with col2:
    st.subheader(f"🎵 Spotify Therapy")

    if latest_mood in ["Stressed", "Sad"]:
        st.markdown(f"We noticed your last journal indicated you were **{latest_mood}**. Here is a calming acoustic playlist to help you relax.")
        playlist_id = "37i9dQZF1DWZqd5JICZI0u" # Peaceful Meditation
    elif latest_mood == "Happy":
        st.markdown(f"You seem to be in a **{latest_mood}** mood! Keep the amazing energy going with this upbeat playlist.")
        playlist_id = "37i9dQZF1DXcBWIGoYBM5M" # Today's Top Hits
    else:
        st.markdown(f"Your mood seems **{latest_mood}**. Here is some great lo-fi focus music for your day.")
        playlist_id = "37i9dQZF1DWWQRwui0ExPn" # Lo-Fi Beats

    # Embed Spotify
    st.markdown(f'<iframe style="border-radius:12px" src="https://open.spotify.com/embed/playlist/{playlist_id}?utm_source=generator" width="100%" height="352" frameBorder="0" allowfullscreen="" allow="autoplay; clipboard-write; encrypted-media; fullscreen; picture-in-picture" loading="lazy"></iframe>', unsafe_allow_html=True)

Overwriting pages/5_Relax.py


In [19]:
from pyngrok import ngrok, conf
import subprocess, time

conf.get_default().auth_token = values["NGROK_AUTHTOKEN"]

# Kill any previous tunnels/streamlit/uvicorn instances from earlier runs in this session
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
get_ipython().system_raw('pkill -f uvicorn || true')
time.sleep(1)

# Launch FastAPI (backend.py) in the background on port 8000 (internal only, not tunneled)
get_ipython().system_raw(
    'uvicorn backend:app --host 0.0.0.0 --port 8000 &'
)
time.sleep(5)  # NLP libs (spaCy model etc.) take a little longer to import

# Launch Streamlit in the background, quietly, on port 8501
get_ipython().system_raw(
    'streamlit run app.py --server.port 8501 --server.headless true '
    '--server.enableCORS false --server.enableXsrfProtection false &'
)
time.sleep(4)  # give both servers a moment to boot

public_url = ngrok.connect(8501, "http")
print(f"🚀 Your app is live at: {public_url}")
print("FastAPI backend is running internally on port 8000 (Streamlit talks to it via localhost).")
print("Open the URL above in your browser. Leave this Colab cell/runtime running to keep it up.")

🚀 Your app is live at: NgrokTunnel: "https://pamperer-twilight-paper.ngrok-free.dev" -> "http://localhost:8501"
FastAPI backend is running internally on port 8000 (Streamlit talks to it via localhost).
Open the URL above in your browser. Leave this Colab cell/runtime running to keep it up.


In [18]:
from pyngrok import ngrok
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
get_ipython().system_raw('pkill -f uvicorn || true')
print("Stopped Streamlit, FastAPI, and closed ngrok tunnel.")

Stopped Streamlit, FastAPI, and closed ngrok tunnel.
